# Day 1-00：環境と保存場所を確認する

**目標**：実験に必要な環境設定やモジュールの挙動の確認をします．
**作業**：
1.  [環境構築ガイド](setup.md) を読みます。
2. 本ノートブックを，ローカルとGoogle Colabの両方で実行し，実行可能かを確認します．同期ズレに注意してください．
> 初回にGoogle Drive for desktopの認証画面が表示されたら、表示された権限を**すべて選択して許可**してください。選択せずに続行すると、Driveの同期やこのNotebookからのファイルアクセスが失敗することがあります。

## 実行前の準備
> **⚠️ 初回に必ず確認：セル冒頭の`DRIVE_PROJECT`と`LOCAL_PROJECT`を、自分の環境の絶対パスに変更してください。**

このセルを最初に実行してください。Driveのマウント、教材フォルダへの移動、Colab用パッケージの準備を自動で行います。
ローカルではカーネルに **Python (jkj1a)** を選びます。


In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parents[1]
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)


## 1. ライブラリとデバイス
CUDAがFalseでもローカルCPUでDay 1を実行できます。GPUを使う回にColabで確認しましょう。

In [ ]:
import sys
import torch
import torchvision
import transformers
import huggingface_hub
import ipywidgets

print("Python:", sys.version.split()[0])
print("Pythonの場所:", sys.executable)
for library in [torch, torchvision, transformers, huggingface_hub, ipywidgets]:
    print(library.__name__, library.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("テンソルの計算:", torch.tensor([1, 2, 3]) * 2)

## 2. スライダーが動くか
スライダーを動かし、下の結果が変わることを確認します。ドラッグ中も値がすぐに更新されます。

In [ ]:
slider = ipywidgets.IntSlider(value=3, min=0, max=10, description="x", continuous_update=True)
square_label = ipywidgets.Label()
def show_square(change=None):
    x = slider.value
    square_label.value = f"x = {x}, x² = {x*x}"
slider.observe(show_square, names="value")
show_square()
display(slider, square_label)


## 3. 保存したファイルを見つける
実行すると `JKJ1A/results/environment-check.txt` ができます。
Colabから実行した場合、同期後にローカルでもこのファイルが見えるか確認してください。
同じPython変数がローカルに移るわけではありません。共有しているのは**保存したファイル**です。

In [ ]:
result_dir = Path("results")
result_dir.mkdir(exist_ok=True)
check_file = result_dir / "environment-check.txt"
check_file.write_text("JKJ1A: file sharing works.\n", encoding="utf-8")
print(check_file.resolve())
print(check_file.read_text(encoding="utf-8"))

## 4. Hugging Faceの準備確認
外部モデルをダウンロードせず、小さなモデルをメモリ内で作って計算します。
学習済みモデルではないので出力に意味はありません。ここではライブラリの動作確認だけをします。

In [ ]:
from transformers import BertConfig, BertModel
config = BertConfig(vocab_size=32, hidden_size=16, num_hidden_layers=1,
                    num_attention_heads=2, intermediate_size=32)
small_model = BertModel(config).eval()
with torch.no_grad():
    outputs = small_model(input_ids=torch.tensor([[1, 2, 3]]))
print("Hugging Face OK:", outputs.last_hidden_state.shape)

## 確認してから次へ
- Pythonが実行されているのは自分のPCか、Colabか？
- `results` の保存場所をエディタ／Google Driveで見つけられるか？
- スライダーの値を変えると結果も変わるか？

次は **01-python-demo.ipynb** です。